In [31]:
!pip install -q -U langchain langchain-text-splitters langchain-community \
    langchain-google-genai langchain-huggingface langchain-chroma \
    chromadb pypdf sentence-transformers


In [32]:
!pip install -q google-generativeai

In [33]:
import os
import json
import re
import getpass

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from google import genai

In [34]:
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

Enter your Google Gemini API key: ··········


In [35]:
# Set your Gemini API key
# Get one free at: https://aistudio.google.com/app/apikey
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

print("API key set")


API key set


In [36]:
from google.colab import files

print("Upload your RESUME (PDF)")
resume_upload = files.upload()
resume_filename = list(resume_upload.keys())[0]

print("\nUpload the JOB DESCRIPTION (PDF or TXT)")
jd_upload = files.upload()
jd_filename = list(jd_upload.keys())[0]

print(f"\nResume file: {resume_filename}")
print(f"Job description file: {jd_filename}")


Upload your RESUME (PDF)


Saving Mahad Rehman Resume.pdf to Mahad Rehman Resume (2).pdf

Upload the JOB DESCRIPTION (PDF or TXT)


Saving AI_Engineer_Job_Description.pdf to AI_Engineer_Job_Description (2).pdf

Resume file: Mahad Rehman Resume (2).pdf
Job description file: AI_Engineer_Job_Description (2).pdf


In [37]:
def extract_text(filepath):
    """Extract text from a PDF or TXT file."""
    if filepath.lower().endswith(".pdf"):
        reader = PdfReader(filepath)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        return text
    else:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

resume_text = extract_text(resume_filename)
jd_text = extract_text(jd_filename)

print("----- Resume preview -----")
print(resume_text[:500])
print("\n----- Job Description preview -----")
print(jd_text[:500])


----- Resume preview -----
MAHAD REHMAN DURRANI
Islamabad, Pakistan
+92 333 9131317|mahadrehman04@gmail.com|/linkedinmahad-rehman|/githubmahadrehmann
EDUCA TION
Bachelor of Science in Computer ScienceExpected June 2026
F AST National University (NUCES), Islamabad
EXPERIENCE
AI Engineer (Apprentice)XFlow Research — June 2025 - April 2026
•Engineering production-gradeAgentic W orkflowsto automate enterprise processes, focusing on scalability.
•Reduced LLM hallucinations and inference costs by implementing strictRAG evaluati

----- Job Description preview -----
AI Engineer
Full-time · On-site / Hybrid · Engineering Department
About the Role
We're looking for an AI Engineer to design, build, and deploy production-grade AI systems — including
LLM-powered applications, agentic workflows, and machine learning pipelines. You'll work closely with
product, data, and infrastructure teams to turn research prototypes into reliable, scalable systems that
real users depend on.
What You'll Do

Design

In [38]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

resume_chunks = splitter.split_text(resume_text)

print(f"Resume split into {len(resume_chunks)} chunks")
for i, chunk in enumerate(resume_chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk[:200])


Resume split into 6 chunks

--- Chunk 1 ---
MAHAD REHMAN DURRANI
Islamabad, Pakistan
+92 333 9131317|mahadrehman04@gmail.com|/linkedinmahad-rehman|/githubmahadrehmann
EDUCA TION
Bachelor of Science in Computer ScienceExpected June 2026
F AST Na

--- Chunk 2 ---
•Reduced LLM hallucinations and inference costs by implementing strictRAG evaluation loops.
•Gathered and translated client requirements into actionable technical specifications.
T eaching Assistant (

--- Chunk 3 ---
PROJECTS
Markly: Autonomous Marketing Agent SystemFYP — FastAPI, LangGraph, Docker
•Engineered a multi-agent marketing system automating the end-to-end campaign lifecycle frommarket
researchtosocial m


In [39]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Quick sanity check
sample_vector = embeddings.embed_query(resume_chunks[0])
print(f"Embedding model loaded. Vector length: {len(sample_vector)}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded. Vector length: 384


In [40]:
# Note: ChromaDB here is in-memory / session-only.
# To persist across sessions, set persist_directory to a Google Drive path (see note at bottom).

vectorstore = Chroma.from_texts(
    texts=resume_chunks,
    embedding=embeddings,
    collection_name="resume_chunks"
)

print("Resume chunks embedded and stored in ChromaDB")


Resume chunks embedded and stored in ChromaDB


In [41]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

relevant_chunks = retriever.invoke(jd_text)

print(f"Retrieved {len(relevant_chunks)} most relevant resume chunks")
for i, doc in enumerate(relevant_chunks):
    print(f"\n--- Relevant Chunk {i+1} ---")
    print(doc.page_content[:200])

retrieved_resume_text = "\n\n".join([doc.page_content for doc in relevant_chunks])


Retrieved 5 most relevant resume chunks

--- Relevant Chunk 1 ---
•A WS Academy Cloud F oundationsby A WS
•Generative AI with LLMsby DeepLearning.AI
•Introduction to Git and GitHubby Google
TECHNICAL SKILLS
LanguagesPython, C/C++, SQL, Java, Bash
AI/MLMachine Learni

--- Relevant Chunk 2 ---
•A WS Academy Cloud F oundationsby A WS
•Generative AI with LLMsby DeepLearning.AI
•Introduction to Git and GitHubby Google
TECHNICAL SKILLS
LanguagesPython, C/C++, SQL, Java, Bash
AI/MLMachine Learni

--- Relevant Chunk 3 ---
•A WS Academy Cloud F oundationsby A WS
•Generative AI with LLMsby DeepLearning.AI
•Introduction to Git and GitHubby Google
TECHNICAL SKILLS
LanguagesPython, C/C++, SQL, Java, Bash
AI/MLMachine Learni

--- Relevant Chunk 4 ---
•Built a ReAct-style RAG agent withLangChainand F AISS, fortified by a strict input validation layer using
the OpenAI Moderation API and Gemini.
•Engineered a 4-stage”LLM-as-a-Judge”pipeline using Lan

--- Relevant Chunk 5 ---
•Built a ReAct-style RAG a

In [42]:
from langchain_core.prompts import PromptTemplate

In [43]:
prompt_template = PromptTemplate(
    input_variables=["job_description", "resume_chunks"],
    template="""
You are an expert ATS (Applicant Tracking System) and technical recruiter.

Compare the RESUME CONTENT below against the JOB DESCRIPTION and evaluate the match.

JOB DESCRIPTION:
{job_description}

RELEVANT RESUME CONTENT:
{resume_chunks}

Analyze the match and respond with ONLY a valid JSON object (no markdown, no extra text)
in exactly this format:

{{
  "ats_score": <integer 0-100>,
  "skill_match": <integer 0-100>,
  "experience_score": <integer 0-100>,
  "education_score": <integer 0-100>,
  "projects_score": <integer 0-100>,
  "formatting_score": <integer 0-100>,
  "matched_skills": ["skill1", "skill2"],
  "missing_skills": ["skill1", "skill2"],
  "suggestions": ["suggestion1", "suggestion2", "suggestion3"]
}}
"""
)

final_prompt = prompt_template.format(
    job_description=jd_text,
    resume_chunks=retrieved_resume_text
)


print("Prompt built")
print(final_prompt[:800])

Prompt built

You are an expert ATS (Applicant Tracking System) and technical recruiter.

Compare the RESUME CONTENT below against the JOB DESCRIPTION and evaluate the match.

JOB DESCRIPTION:
AI Engineer
Full-time · On-site / Hybrid · Engineering Department
About the Role
We're looking for an AI Engineer to design, build, and deploy production-grade AI systems — including
LLM-powered applications, agentic workflows, and machine learning pipelines. You'll work closely with
product, data, and infrastructure teams to turn research prototypes into reliable, scalable systems that
real users depend on.
What You'll Do

Design and build AI-powered features, including LLM integrations, RAG pipelines, and agentic
workflows.

Develop, fine-tune, and evaluate machine learning models for production use cases.




In [44]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",  # ✅ Uses less quota
    temperature=0.2
)
response = llm.invoke(final_prompt)
raw_output = response.content

print("----- Raw Gemini Output -----")
print(raw_output)


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


----- Raw Gemini Output -----
[{'type': 'text', 'text': '{\n  "ats_score": 76,\n  "skill_match": 90,\n  "experience_score": 60,\n  "education_score": 65,\n  "projects_score": 88,\n  "formatting_score": 55,\n  "matched_skills": [\n    "Python",\n    "RAG Pipelines",\n    "LangChain",\n    "CrewAI",\n    "Vector Databases (FAISS)",\n    "FastAPI",\n    "Docker",\n    "Kubernetes",\n    "AWS",\n    "GCP",\n    "OpenAI API",\n    "Agentic Workflows",\n    "LangSmith"\n  ],\n  "missing_skills": [\n    "3+ years professional software engineering work experience",\n    "Anthropic API",\n    "Model fine-tuning implementation details",\n    "Automated retraining pipelines"\n  ],\n  "suggestions": [\n    "Remove duplicate text blocks and formatting artifacts to allow ATS systems to parse the document cleanly.",\n    "Include explicit professional work history with dates and titles to prove the required 3+ years of engineering and 1+ years of production ML experience.",\n    "Add concrete bullet 

In [46]:
import re
import json

def parse_json_response(text):
    """Extract a JSON object from the model's response, even if wrapped in markdown fences."""
    # Handle case where content is a list of blocks instead of a string
    if isinstance(text, list):
        text = "".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in text
        )
    cleaned = re.sub(r"```json|```", "", text).strip()
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model output")
    return json.loads(match.group(0))

analysis = parse_json_response(raw_output)
print("Parsed JSON:")
print(json.dumps(analysis, indent=2))

Parsed JSON:
{
  "ats_score": 76,
  "skill_match": 90,
  "experience_score": 60,
  "education_score": 65,
  "projects_score": 88,
  "formatting_score": 55,
  "matched_skills": [
    "Python",
    "RAG Pipelines",
    "LangChain",
    "CrewAI",
    "Vector Databases (FAISS)",
    "FastAPI",
    "Docker",
    "Kubernetes",
    "AWS",
    "GCP",
    "OpenAI API",
    "Agentic Workflows",
    "LangSmith"
  ],
  "missing_skills": [
    "3+ years professional software engineering work experience",
    "Anthropic API",
    "Model fine-tuning implementation details",
    "Automated retraining pipelines"
  ],
  "suggestions": [
    "Remove duplicate text blocks and formatting artifacts to allow ATS systems to parse the document cleanly.",
    "Include explicit professional work history with dates and titles to prove the required 3+ years of engineering and 1+ years of production ML experience.",
    "Add concrete bullet points highlighting hands-on model fine-tuning or evaluation against custom

## Section 10 — Calculate Final ATS Score

In [47]:
def calculate_final_score(analysis):
    weights = {
        "skill_match": 0.35,
        "experience_score": 0.25,
        "education_score": 0.15,
        "projects_score": 0.15,
        "formatting_score": 0.10
    }

    weighted_sum = sum(analysis.get(key, 0) * weight for key, weight in weights.items())
    return round(weighted_sum)

# Use Gemini's own ats_score if present, otherwise compute from weighted components
final_ats_score = analysis.get("ats_score") or calculate_final_score(analysis)

print(f"Final ATS Score: {final_ats_score}%")


Final ATS Score: 76%


## Section 11 — Print the Dashboard

In [48]:
def print_dashboard(analysis, final_score):
    line = "=" * 45
    print(line)
    print("   AI ATS RESUME ANALYZER - DASHBOARD")
    print(line)
    print(f"ATS Score      : {final_score}%")
    print(f"Skill Match    : {analysis.get('skill_match', 0)}%")
    print(f"Experience     : {analysis.get('experience_score', 0)}%")
    print(f"Education      : {analysis.get('education_score', 0)}%")
    print(f"Projects       : {analysis.get('projects_score', 0)}%")
    print(f"Formatting     : {analysis.get('formatting_score', 0)}%")
    print()
    print("Matched Skills")
    for skill in analysis.get("matched_skills", []):
        print(f"  + {skill}")
    print()
    print("Missing Skills")
    for skill in analysis.get("missing_skills", []):
        print(f"  - {skill}")
    print()
    print("Suggestions")
    for suggestion in analysis.get("suggestions", []):
        print(f"  * {suggestion}")
    print(line)

print_dashboard(analysis, final_ats_score)


   AI ATS RESUME ANALYZER - DASHBOARD
ATS Score      : 76%
Skill Match    : 90%
Experience     : 60%
Education      : 65%
Projects       : 88%
Formatting     : 55%

Matched Skills
  + Python
  + RAG Pipelines
  + LangChain
  + CrewAI
  + Vector Databases (FAISS)
  + FastAPI
  + Docker
  + Kubernetes
  + AWS
  + GCP
  + OpenAI API
  + Agentic Workflows
  + LangSmith

Missing Skills
  - 3+ years professional software engineering work experience
  - Anthropic API
  - Model fine-tuning implementation details
  - Automated retraining pipelines

Suggestions
  * Remove duplicate text blocks and formatting artifacts to allow ATS systems to parse the document cleanly.
  * Include explicit professional work history with dates and titles to prove the required 3+ years of engineering and 1+ years of production ML experience.
  * Add concrete bullet points highlighting hands-on model fine-tuning or evaluation against custom metrics.


---
## Optional — Persist ChromaDB to Google Drive
By default, ChromaDB in this notebook is **in-memory** and disappears when the Colab
runtime disconnects. If you want it to persist across sessions, mount Google Drive and
set a `persist_directory`:

```python
from google.colab import drive
drive.mount('/content/drive')

vectorstore = Chroma.from_texts(
    texts=resume_chunks,
    embedding=embeddings,
    collection_name="resume_chunks",
    persist_directory="/content/drive/MyDrive/ats_chroma_db"
)
```

For a learning project, the in-memory version above is perfectly fine — just re-run the
notebook top to bottom each session.

---
### Next steps
Once this works well in Colab, move the logic into a proper Python project
(`app.py`, `requirements.txt`, `src/` folder) and push it to GitHub for your portfolio.
